# Capon's Beamformer 

In a previous section I went over how to make a **very basic** beamformer using the simple geometry of a phased array and the expected angle of arrival. 

However it turns out that we can do much better than this traditional beamformer using techniques refered to as super resolution.

I was one of the primary developers for [**Hack-A-Sat 4**](https://hackasat.com/hackasat4/) and during our qualifying round I authored a challenge called ["Signals from Trisolaris"](https://github.com/cromulencellc/hackasat-qualifier-2023/tree/main/challenges/5_Van_Halen_Radiation_Belt/2_Signals_From_Trisolaris) in this challenge I presented the user with a signal from an environment where a 2 dimensional planar phased array had received signals at the same time from five different azimuth and elevations that corresponded to five different stars in the night sky.

To solve this challenge the players had to use both signal processing and astrodynamics:
1. Signal Processing: Devise a beam former that could search azimuth and elevation **and** identify the incoming direction of each signal 
2. Astrodynamics: Translate those azimuth and elevations into the specific stars that the signal came from.


In this lab I'll focus on the signal processing section. We will not concern ourselves with astrodynamics which I can cover in a different lab.

Fun fact one of our players said: "I thought I understood phased arrays until I played this CTF - I stand corrected"

That's always the sign of a great ctf challenge!




## 2D Planar arrays 

In the previous section the antennas sat on a line, so the array could only tell us one angle $\theta$ off boresight. To find a direction in the sky we need two angles, azimuth and elevation, which means we need to spread the antennas out in two dimensions. A **planar array** is a grid of antennas laid out on a flat surface.

In this lab the array is lying flat on the ground looking straight up at the sky. Each element $n$ has a position in the plane of the array measured in wavelengths:
- $H_n$ the horizontal position, pointing **East**
- $V_n$ the vertical position, pointing **North**

We are using an $8 \times 8$ grid with $0.25\lambda$ spacing (`ANTENNA_H_GRID`, `ANTENNA_V_GRID`), so 64 elements total.

### Direction of arrival

A signal arriving from azimuth $az$ (measured from North towards East) and elevation $el$ (measured up from the horizon) travels along the unit vector, written in East, North, Up:

$$
\mathbf{u}(az, el) = \begin{bmatrix} \cos(el)\sin(az) \\ \cos(el)\cos(az) \\ \sin(el) \end{bmatrix}
$$

Just like the linear array, the extra distance the wave has to travel to reach each element is the element's position projected onto that direction. Since every element sits at Up $= 0$, the third term drops out:

$$
\Delta P_n = \mathbf{p}_n \cdot \mathbf{u} = H_n \cos(el)\sin(az) + V_n \cos(el)\cos(az)
$$

This is what `azElToWaveDelay` computes (it builds $\mathbf{u}$ with rotations instead, but you get the same answer). Because the positions are already in wavelengths, the phase at each element is just

$$
\Delta\phi_n = 2\pi \, \Delta P_n
$$

### Steering vector

Stacking the phases for all $Z$ elements gives the 2D steering vector, which is what a signal from $(az, el)$ "looks like" across the array:

$$
\mathbf{a}(az, el) = \begin{bmatrix} e^{j 2\pi \Delta P_0} & e^{j 2\pi \Delta P_1} & \cdots & e^{j 2\pi \Delta P_{Z-1}} \end{bmatrix}^T
$$

In the code this is `v0` inside `mvdr_2d`, and the signal generator uses the same math to put each source onto the antennas.

### Things to keep in mind
- **Spacing:** the same $\lambda/2$ Nyquist rule applies, now in both directions. Our $0.25\lambda$ spacing is well under that, so there are no grating lobes.
- **Small aperture:** the whole array is only $1.75\lambda$ across, so a conventional beamformer would have a very wide beam. That's the motivation for MVDR below.
- **Resolution changes with direction:** near the horizon ($el \approx 0$) $\cos(el)$ barely changes with elevation, so elevation is hard to pin down. You can see this as the tall vertical streaks on the low elevation sources in the heatmap. Near zenith ($el \approx 90$) $\cos(el) \approx 0$, so azimuth becomes hard to pin down instead.
- **Above vs below:** a flat array can't tell $+el$ from $-el$ because $\cos$ is symmetric, which is why we only search $0$ to $90$ degrees.

## The minimum variance distortionless response (MVDR)

- MVDR or Capons beam former 

### Signal model

At sample $k$ the array gives us a $Z \times 1$ vector $\mathbf{x}[k]$, one complex sample per antenna. With $K$ sources each sending its own signal $s_i[k]$, plus noise $\mathbf{n}[k]$ at each antenna:

$$
\mathbf{x}[k] = \sum_{i=1}^{K} \mathbf{a}(az_i, el_i) \, s_i[k] + \mathbf{n}[k]
$$

This is exactly what the signal generator builds: `signalExpanded` is each term of the sum, and `signal` is $\mathbf{x}$ once they're added together with the noise.

### Spatial covariance

Everything MVDR knows about the environment comes from the **spatial covariance matrix**, which tells us how correlated each pair of antennas is:

$$
\mathbf{R} = E\left[\mathbf{x}\mathbf{x}^H\right] \approx \frac{1}{M}\sum_{k=0}^{M-1} \mathbf{x}[k]\,\mathbf{x}[k]^H
$$

This is `spatial_covariance`, using $M$ samples. $\mathbf{R}$ is $Z \times Z$ and contains every source and the noise all mixed together.

### Conventional vs MVDR

Any beamformer is a set of weights $\mathbf{w}$, and its output is $y[k] = \mathbf{w}^H \mathbf{x}[k]$. The average output power is

$$
P = E\left[|y|^2\right] = \mathbf{w}^H \mathbf{R} \, \mathbf{w}
$$

The conventional beamformer from the last section just uses $\mathbf{w} = \mathbf{a}/Z$. Its weights are fixed, so energy from other sources leaks in through the sidelobes and the beam width is set by the array size.

MVDR instead picks the weights based on the data. For each look direction $\mathbf{a}$ it asks:

> Minimize the output power $\mathbf{w}^H\mathbf{R}\,\mathbf{w}$ (**minimum variance**), while keeping a gain of exactly 1 in the look direction, $\mathbf{w}^H\mathbf{a} = 1$ (**distortionless response**).

The only way to lower the output power without touching the look direction is to cancel everything else, so MVDR automatically places nulls on the other sources.

### Solving for the weights

Using a Lagrange multiplier $\mu$:

$$
L(\mathbf{w}) = \mathbf{w}^H\mathbf{R}\,\mathbf{w} + \mu\left(\mathbf{w}^H\mathbf{a} - 1\right)
$$

Setting the derivative with respect to $\mathbf{w}^*$ to zero gives $\mathbf{R}\,\mathbf{w} + \mu\,\mathbf{a} = 0$, so $\mathbf{w} = -\mu\,\mathbf{R}^{-1}\mathbf{a}$. Plugging that into the constraint to solve for $\mu$ gives the MVDR weights:

$$
\mathbf{w} = \frac{\mathbf{R}^{-1}\mathbf{a}}{\mathbf{a}^H\mathbf{R}^{-1}\mathbf{a}}
$$

And putting those weights back into $\mathbf{w}^H\mathbf{R}\,\mathbf{w}$ gives the power coming from that direction:

$$
P_{MVDR}(az, el) = \frac{1}{\mathbf{a}^H(az,el)\,\mathbf{R}^{-1}\,\mathbf{a}(az,el)}
$$

In the code `mvdr` computes `alpha` $= P_{MVDR}$ and `w`, `mvdr_2d` builds $\mathbf{a}$ for one $(az, el)$, and `mvdr_search` sweeps the whole sky to build the heatmap. Peaks in $P_{MVDR}$ are the sources.

### Things to keep in mind
- **Invert once:** $\mathbf{R}^{-1}$ only has to be computed once. Each look direction after that is just one quadratic form, which is why we pass `Rinv` around.
- **Enough samples:** $\mathbf{R}$ is only invertible if $M \geq Z$, and in practice you want a good margin. We use $M = 100$ for $Z = 64$ antennas.
- **Noise helps:** with no noise $\mathbf{R}$ has rank $K = 5$ and can't be inverted at all. Real systems often add a small "diagonal loading" term $\mathbf{R} + \epsilon\mathbf{I}$ for the same reason.
- **Independent sources:** MVDR assumes the sources are uncorrelated. If two are coherent (e.g. multipath copies of the same signal) it treats one as interference and cancels it, which is why the generator gives every source its own random symbols.


Notice when you run the MVDR beamformer and compare it to the regular broadside beamformer the results are much tighter in az/el space and the signals are very distinct.

In [35]:
%pip install -q scipy plotly nbformat ipywidgets


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
import numpy as np 
from scipy.spatial.transform import Rotation as R
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment

import plotly.graph_objects as go

In [37]:
# Signal specs 

# Signal incoming angles in (AZ,EL) in degrees 
sourcesAzEl = [ (272.5,45.2) , (20.7,30.3) , (173.4,65.9) , (111.24,10.11) , ( 310.0,22.1 )]
frequency = 10.0e9 # Signal frequency in HZ 
sampleRate = 100.0e6 
snrDb = 10.0 # Per-antenna SNR of each source vs. the receiver noise


# Horizontal and vertical antenna positions in wavelengths
# Note: the antennas are spaced in H/V less than 0.5 wavelengths because we need to think about distance along diag directiosn too
ANTENNA_V_GRID = np.arange(-1,1,0.25)
ANTENNA_H_GRID = np.arange(-1,1,0.25)

In [38]:
# Signal generator 
N = 10000 # Samples to generate 
locations = [] 
for V in ANTENNA_V_GRID:
    for H in ANTENNA_H_GRID: 
        loc = (H,V)
        locations.append(loc)
Z = len( locations )
K = len( sourcesAzEl)

rng = np.random.default_rng(0)

# Path difference (in wavelengths) of each source at each antenna, shape (Z,K)
# H is East and V is North, so project each location onto the unit vector toward the source
azRad = np.radians( [ az for az,el in sourcesAzEl ] )
elRad = np.radians( [ el for az,el in sourcesAzEl ] )
east  = np.cos(elRad) * np.sin(azRad)
north = np.cos(elRad) * np.cos(azRad)
locArray = np.array( locations )
waveDelays = np.outer( locArray[:,0], east ) + np.outer( locArray[:,1], north )
steering = np.exp( 1j * 2*np.pi * waveDelays )

# Each source transmits its own independent QPSK symbols at baseband, shape (N,K)
# Independent sources matter: MVDR can't separate coherent (identical) signals
symbols = np.exp( 1j * np.pi/2 * rng.integers(0, 4, (N,K)) )

# (N,Z,K) set of signals. note the third dimension is for each source
signalExpanded = symbols[:, np.newaxis, :] * steering[np.newaxis, :, :]
# sum together so we have (N,Z) one set of N signals for each antenna 
signal = np.sum( signalExpanded, axis=2 )

# Add complex white noise at each antenna. This also keeps the covariance invertible
noiseStd = 10**( -snrDb / 20 )
signal = signal + noiseStd/np.sqrt(2) * ( rng.standard_normal((N,Z)) + 1j*rng.standard_normal((N,Z)) )

In [39]:
# Capons beamformer helpers 
def spatial_covariance( signals , N ):
    # For a vector of L signals
    L = len(signals)
    # Calculate the spatial covariance based on N samples

    # Verify that we have enough 
    R = np.zeros(  (L,L) )
    for k in range(0,N):
        X = np.matrix( signals[:,k] ).transpose()
        Rk = X @ X.H 
        R =  R + Rk 
    R = R / N
    return R
def columnRotHelper( axis , angle ):
    return np.squeeze(R.from_euler( axis, [angle],degrees=True).as_matrix()).transpose()

def azElToWaveDelay( azDegrees, elDegrees , locations ):
    azRot = columnRotHelper("x", -azDegrees )
    elRot = columnRotHelper("y",-(90.0-elDegrees))
    UEN_2_LOS = np.matmul( elRot,  azRot )
    LOS_2_UEN  = UEN_2_LOS.transpose()
    los_LOS_Frame = np.array( [1,0,0] )
    los_UEN_FRAME = np.matmul( LOS_2_UEN , los_LOS_Frame )
    waveDistances = []
    for idx,loc in  enumerate(locations):
        UEN = np.array( [0 , loc[0], loc[1] ] ) 
        waveDistance = np.dot( UEN , los_UEN_FRAME) # wavelengths
        waveDistances.append(waveDistance)
    return np.array( waveDistances ) 
def mvdr( Rinv , v0 , locations ):
    # Capon gain to enforce unity
    alpha =( 1.0 / ( v0.H @ Rinv  @ v0 ) )
    alpha =  alpha.flatten().item(0)
    w = alpha * (Rinv @ v0)
    return alpha, w 
def mvdr_2d( Rinv, az,el, locations ):
    waveLengths = azElToWaveDelay( az,el,locations)
    phases = 2 *np.pi*waveLengths 
    v  =  np.exp(  1j* phases , dtype=np.complex64)
    v0 = np.matrix( v ).transpose()
    alpha,w = mvdr( Rinv ,v0 ,locations )
    pow = np.real( np.abs( alpha ) ) 
    return pow, w 

def mvdr_search( Rinv , arrayLocation, stepDeg=0.1 ):
    azSearch = np.arange(0,360, stepDeg )
    elSearch = np.arange( 0,90 ,stepDeg )
    P = np.zeros( (len(azSearch) , len(elSearch)) , dtype=np.float32)
    for k,az in enumerate(azSearch):
        for z,el in enumerate( elSearch ):
            pow,w = mvdr_2d( Rinv, az,el , arrayLocation )
            P[k][z] = pow 

    return azSearch,elSearch, P


def broadside_search( signal, arrayLocation, stepDeg=0.1 ):
    azSearch = np.arange(0,360, stepDeg )
    elSearch = np.arange( 0,90 ,stepDeg )
    P = np.zeros( (len(azSearch) , len(elSearch)) , dtype=np.float32)
    for k,az in enumerate(azSearch):
        for z,el in enumerate( elSearch ):
            pow,w = broadside_2d( signal, az,el , arrayLocation )
            P[k][z] = pow 

    return azSearch,elSearch, P

def broadside_2d( signal, az, el, arrayLocation ):
    # Conventional beamformer: steer to (az,el) with fixed weights w = a/Z
    waveLengths = azElToWaveDelay( az,el,arrayLocation )
    a = np.exp( 1j * 2*np.pi*waveLengths )
    w = a / len(arrayLocation)
    # Output y[k] = w^H x[k] for every sample, signal is (N,Z)
    y = signal @ np.conj(w)
    pow = np.mean( np.abs(y)**2 )
    return pow, w 

In [40]:
# Ploting helpers 

def plotSearch( azSearch, elSearch, PdB, title="" ):
    
    # az,el, color plot of P 
    fig = go.Figure()
    fig.add_trace( go.Heatmap( x=azSearch, y=elSearch, z=PdB.T, colorscale="Magma", zmin=-30, zmax=0,
                               colorbar=dict(title="Power<br>(dB, normalized)"),
                               hovertemplate="az %{x:.1f}°<br>el %{y:.1f}°<br>%{z:.1f} dB<extra></extra>" ) )
    fig.add_trace( go.Scatter( x=[az for az,el in sourcesAzEl], y=[el for az,el in sourcesAzEl],
                               mode="markers+text", name="True sources",
                               text=[f"({az},{el})" for az,el in sourcesAzEl], textposition="top right",
                               textfont=dict(color="white"),
                               marker=dict(size=16, color="rgba(0,0,0,0)", line=dict(color="white", width=2)),
                               hovertemplate="true source<br>az %{x}°<br>el %{y}°<extra></extra>" ) )
    fig.update_layout( title=f"{title} spectrum (circles = true sources)", width=900, height=450,
                       template="plotly_white", showlegend=False,
                       xaxis=dict(title="Azimuth (deg)", range=[0,360]),
                       yaxis=dict(title="Elevation (deg)", range=[0,90]) )
    fig.show()

def plotArray( locations ):
    # A drawing of the array 
    H, V = np.array( locations ).T
    fig = go.Figure()
    fig.add_trace( go.Scatter( x=H, y=V, mode="markers", marker=dict(symbol="square", size=14, color="#2a6fdb"),
                           hovertemplate="H %{x} λ<br>V %{y} λ<extra></extra>" ) )
    fig.update_layout( title=f"{len(ANTENNA_H_GRID)} x {len(ANTENNA_V_GRID)} planar array ({Z} elements)",
                   width=500, height=500, template="plotly_white",
                   xaxis=dict(title="Horizontal position (wavelengths)"),
                   yaxis=dict(title="Vertical position (wavelengths)", scaleanchor="x", scaleratio=1) )
    fig.show()

def checkMatches( azSearch, elSearch, P  ):
    
    # A peak is a point that is the max of its neighborhood. Azimuth wraps at 360, elevation doesn't
    neighborhood = 9
    isPeak = P == maximum_filter( P, size=neighborhood, mode=("wrap","nearest") )
    peakIdx = np.argwhere( isPeak )
    # Keep the K strongest peaks, one per source
    peakIdx = peakIdx[ np.argsort( -P[isPeak] ) ][:K]
    solvedAzEl = [ (azSearch[a], elSearch[e]) for a,e in peakIdx ]
    
    def unitVector( az, el ):
        az, el = np.radians(az), np.radians(el)
        return np.array( [ np.cos(el)*np.sin(az), np.cos(el)*np.cos(az), np.sin(el) ] )
    
    def angleBetween( azEl1, azEl2 ):
        return np.degrees( np.arccos( np.clip( unitVector(*azEl1) @ unitVector(*azEl2), -1, 1 ) ) )
    
    # Pair each true source with a solved peak, minimizing the total angular error
    cost = np.array( [ [ angleBetween(t,s) for s in solvedAzEl ] for t in sourcesAzEl ] )
    truthIdx, solvedIdx = linear_sum_assignment( cost )
    
    print( f"{'Solved (az,el)':>14} | {'Truth (az,el)':>13} | {'Az err':>7} | {'El err':>7} | {'Angle err':>9}" )
    print( "-"*63 )
    for t,s in zip( truthIdx, solvedIdx ):
        (tAz,tEl), (sAz,sEl) = sourcesAzEl[t], solvedAzEl[s]
        azErr = (sAz - tAz + 180) % 360 - 180 # wrap to +/-180
        elErr = sEl - tEl
        print( f"({sAz:6.1f},{sEl:5.1f}) | ({tAz:5.1f},{tEl:5.1f}) | {azErr:+7.1f} | {elErr:+7.1f} | {cost[t,s]:9.2f}" )

In [41]:
# DO the solution

M = 100 
# spatial_covariance expects one row per antenna, so pass (Z,N)
Rinv = np.linalg.inv( spatial_covariance( signal.T, M ) )

# 1 degree takes a few seconds; 0.1 degree is ~100x slower
azSearch, elSearch, P = mvdr_search( Rinv, arrayLocation=locations, stepDeg=1.0 )
# Conventional beamformer on the same M samples for comparison
azSearchB, elSearchB, PB = broadside_search( signal[:M], arrayLocation=locations, stepDeg=1.0 )
PdB = 10*np.log10( P / P.max() )
PBdB = 10*np.log10( PB / PB.max() )


plotArray(locations)
plotSearch( azSearch,elSearch,PdB, title="Capon / MVDR" )
plotSearch( azSearchB,elSearchB,PBdB, title="Conventional (broadside)" )
checkMatches( azSearch,elSearch,P)

Solved (az,el) | Truth (az,el) |  Az err |  El err | Angle err
---------------------------------------------------------------
( 272.0, 45.0) | (272.5, 45.2) |    -0.5 |    -0.2 |      0.41
(  21.0, 30.0) | ( 20.7, 30.3) |    +0.3 |    -0.3 |      0.40
( 174.0, 66.0) | (173.4, 65.9) |    +0.6 |    +0.1 |      0.26
( 111.0, 10.0) | (111.2, 10.1) |    -0.2 |    -0.1 |      0.26
( 310.0, 22.0) | (310.0, 22.1) |    +0.0 |    -0.1 |      0.10
